# **Collected Data**

In [4]:
import os
import tarfile
import gzip
import shutil
import pandas as pd
import pickle

path = "/home/uynk/Belgeler/Analyze_Projects/AliBaba_GenAI_Dataset/v2026_GenAI"

dataframes = {}

def is_gzip(file_path):
    """Checks whether a file is truly in gzip format."""
    try:
        with open(file_path, "rb") as f:
            return f.read(2) == b"\x1f\x8b"
    except:
        return False

for file_name in os.listdir(path):
    if file_name.endswith(".tar.gz"):
        file_path = os.path.join(path, file_name)
        
        try:
            with tarfile.open(file_path, "r:gz") as tar:
                tar.extractall(path=path)
                extracted_files = tar.getnames()
                
                for extracted in extracted_files:
                    full_path = os.path.join(path, extracted)
                    if not os.path.isfile(full_path): 
                        continue

                    if is_gzip(full_path):
                        decompressed_path = full_path + ".csv"
                        print(f"📦 Gzip detected in content: {extracted} -> Extracting...")
                        try:
                            with gzip.open(full_path, "rb") as f_in:
                                with open(decompressed_path, "wb") as f_out:
                                    shutil.copyfileobj(f_in, f_out)
                            full_path = decompressed_path
                        except Exception as e:
                            print(f"❌ Gunzip error ({extracted}): {e}")
                            continue

                    try:
                        df = pd.read_csv(full_path)
                        if df.empty:
                            print(f"⚠️ WARNING: {os.path.basename(full_path)} is EMPTY!")
                        else:
                            df_name = os.path.basename(full_path)
                            dataframes[df_name] = df
                            print(f"✅ OK (From Tar): {df_name} successfully added to the dictionary.")
                    except Exception as e:
                        print(f"❌ ERROR: Could not read {os.path.basename(full_path)}: {e}")
                        
        except tarfile.ReadError:
            print(f"🔄 INFO: {file_name} is not a tar archive. Reading directly with pandas...")   
            try:
                # Reading the file directly as in your example
                df = pd.read_csv(file_path, compression='gzip')
                
                if df.empty:
                    print(f"⚠️ WARNING: {file_name} is EMPTY!")
                else:
                    dataframes[file_name] = df
                    print(f"✅ OK (Directly): {file_name} read and added to the dictionary.")
                    
            except Exception as e:
                print(f"💥 CRITICAL ERROR: {file_name} could not be read as a tar or directly: {e}")
                
        except Exception as e:
            print(f"💥 UNEXPECTED ERROR: A problem occurred while processing {file_name}: {e}")

print(f"\n--- All operations completed! A total of {len(dataframes)} dataframes were loaded. ---")

✅ OK (From Tar): pipeline_inference_data_anon.csv successfully added to the dictionary.
❌ ERROR: Could not read ._data_trace_processed.csv: 'utf-8' codec can't decode byte 0x90 in position 211: invalid start byte
✅ OK (From Tar): data_trace_processed.csv successfully added to the dictionary.
✅ OK (From Tar): queue_size_raw_anon.csv successfully added to the dictionary.
✅ OK (From Tar): controlnet_latency_data_anon.csv successfully added to the dictionary.
✅ OK (From Tar): lora_request_trace.csv successfully added to the dictionary.
✅ OK (From Tar): lora_update_latency_anon.csv successfully added to the dictionary.
✅ OK (From Tar): pod_memory_util_anon.csv successfully added to the dictionary.
✅ OK (From Tar): model_predict_data_anon.csv successfully added to the dictionary.
✅ OK (From Tar): queue_rt_raw_anon.csv successfully added to the dictionary.
✅ OK (From Tar): pod_gpu_duty_cycle_anon.csv successfully added to the dictionary.
✅ OK (From Tar): pipeline_update_latency_anon.csv succe

# Data Preprocessing

In [5]:
def new_feature(df_dict):
    for var in df_dict:
        if "timestamp_anon" in df_dict[var].columns:
            
            df_dict[var]["real_time"] = pd.to_datetime(df_dict[var]["timestamp_anon"], unit="s")
            df_dict[var]["real_time_CST"] = df_dict[var]["real_time"] + pd.Timedelta(hours=8)
            
            df_dict[var].sort_values("real_time", inplace=True)
            df_dict[var].reset_index(drop=True, inplace=True)
        
            print(f"{var}: Created 'real_time' and 'real_time_CST', and sorted data.")
        
        else:
            print(f"{var}: Column 'timestamp_anon' not found")

new_feature(dataframes)


path2 = "/home/uynk/Belgeler/Analyze_Projects/AliBaba_GenAI_Dataset"
os.makedirs(path2, exist_ok=True)

file_path = f'{path2}/dataframes.pkl'
with open(file_path, 'wb') as file:
    pickle.dump(dataframes, file)

print(f"Successfully saved to '{file_path}'.")

pipeline_inference_data_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
data_trace_processed.csv: Column 'timestamp_anon' not found
queue_size_raw_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
controlnet_latency_data_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
lora_request_trace.csv: Column 'timestamp_anon' not found
lora_update_latency_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
pod_memory_util_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
model_predict_data_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
queue_rt_raw_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
pod_gpu_duty_cycle_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
pipeline_update_latency_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
qps.csv: Created 'real_time' and 'real_time_CST', and sorted data.
pod_gpu_memory_used_bytes_ano